# Extract genotypes by keep-file and variant list

This notebook builds and runs a PLINK command that:
- keeps samples listed in a keep file (person IDs)
- extracts variants listed in a variant file
- supports variant file formats:
  - `.txt`: one 1-indexed variant position per line (requires `--chr`)
  - `.bed`: BED intervals converted to variant IDs via `bcftools` (requires `--vcf`)

> Note: For `.txt`, positions are interpreted as 1-indexed base-pair positions, matching PLINK's `--from-bp/--to-bp` logic.

In [9]:
import os
import shlex
import subprocess
from pathlib import Path

# ---------- USER INPUTS ----------
KEEP_FILE = "/home/jupyter/repos/aou_disease_prediction_2/ckd_patients_for_plink.txt"        # e.g. "/path/to/keep.txt" with columns: FID IID (or person_id mapped to both)
VARIANTS_FILE = "/home/jupyter/repos/aou_disease_prediction_2/chr4_pkd_variants.bed"    # e.g. "/path/to/variants.txt" (1-indexed positions) OR "/path/to/regions.bed"

# Input genotype source: choose ONE mode
PLINK_PFILE = "/home/jupyter/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/pgen/clinvar.chr4"      # e.g. "/path/to/genotype_prefix" (expects .bed/.bim/.fam)
VCF_FILE = ""         # optional, needed for BED-region workflow to build variant ID list

# Required when VARIANTS_FILE ends with .txt (1-indexed positions)
CHR = "4"              # e.g. "21"

# Output prefix
OUT_PREFIX = "plink_subset"

# Temporary working files
TMP_DIR = Path("tmp_plink_extract")
TMP_DIR.mkdir(exist_ok=True)


def run(cmd: str):
    print("\n$", cmd)
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")
    return completed


def ensure_file(path: str, label: str):
    if not path or not Path(path).exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def normalize_keep_file(keep_file: str, out_keep: Path) -> Path:
    """
    Accepts either:
    - 1-column person_id
    - 2-column FID IID
    Produces a 2-column keep file for PLINK.
    """
    with open(keep_file, "r", encoding="utf-8") as fin, open(out_keep, "w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) == 1:
                pid = parts[0]
                fout.write(f"{pid} {pid}\n")
            else:
                fout.write(f"{parts[0]} {parts[1]}\n")
    return out_keep


def variants_txt_to_extract_ranges(variants_txt: str, chr_value: str, out_ranges: Path) -> Path:
    """
    Converts 1-indexed positions in .txt into PLINK --range format:
    CHR START END
    (single-base ranges: START==END)
    """
    if not chr_value:
        raise ValueError("CHR is required when using a .txt variants file.")

    with open(variants_txt, "r", encoding="utf-8") as fin, open(out_ranges, "w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            pos = int(line)
            if pos < 1:
                raise ValueError(f"Position must be 1-indexed and >= 1. Got: {pos}")
            fout.write(f"{chr_value} {pos} {pos}\n")
    return out_ranges


def bed_to_variant_ids(bed_file: str, vcf_file: str, out_ids: Path) -> Path:
    """
    Uses bcftools to query IDs overlapping BED regions.
    Requires a VCF with indexed .tbi/.csi and valid ID field.
    """
    ensure_file(vcf_file, "VCF_FILE")
    cmd = (
        f"bcftools view -R {shlex.quote(bed_file)} {shlex.quote(vcf_file)} | "
        "bcftools query -f '%ID\\n' | awk 'NF' | sort -u > "
        f"{shlex.quote(str(out_ids))}"
    )
    run(cmd)
    if out_ids.stat().st_size == 0:
        raise ValueError("No variant IDs found from BED regions in provided VCF.")
    return out_ids


In [7]:
genomic_data_path = "/home/jupyter/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled"
plink_bed = f"{genomic_data_path}/v8/wgs/short_read/snpindel/clinvar/plink_bed/chr4.bed"
plink_bim = f"{genomic_data_path}/v8/wgs/short_read/snpindel/clinvar/plink_bed/chr4.bim"
plink_fam = f"{genomic_data_path}/v8/wgs/short_read/snpindel/clinvar/plink_bed/chr4.fam"

In [12]:
!head -n 3 "{plink_bim}"

chr4	chr4:85799:A:G	0.0	85799	G	A
chr4	chr4:85805:A:C	0.0	85805	C	A
chr4	chr4:85866:A:G	0.0	85866	G	A


In [15]:
# Write bash code to find a specific number in plink_bim
!awk '$4 == 85799 {print $2}' "$plink_bim"


In [9]:
%%bash

KEEP_FILE="/home/jupyter/repos/aou_disease_prediction_2/ckd_patients_for_plink.txt"        # e.g. "/path/to/keep.txt" with columns: FID IID (or person_id mapped to both)
VARIANTS_FILE="/home/jupyter/repos/aou_disease_prediction_2/chr4_pkd_variants.bed"    # e.g. "/path/to/variants.txt" (1-indexed positions) OR "/path/to/regions.bed"
PLINK_PFILE="/home/jupyter/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/pgen/exome.chr4"      # e.g. "/path/to/genotype_prefix" (expects .bed/.bim/.fam)
PLINK_BFILE="/home/jupyter/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/bgen/chr4.bgen"      # e.g. "/path/to/genotype_prefix" (expects .bed/.bim/.fam)

/opt/workbench-tools/binaries/bin/plink2 --pfile "$PLINK_PFILE" \
  --keep "$KEEP_FILE" \
  --extract bed1 "$VARIANTS_FILE" \
  --make-pgen \
  --out ./plink_outputs/subset_out

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/


(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3


Logging to subset_out.log.


Options in effect:


  --extract bed1 /home/jupyter/repos/aou_disease_prediction_2/chr4_pkd_variants.bed


...

--extract bed1: 1553124 variants excluded.


--keep: 0 samples remaining.


Error: No samples remaining after main filters.


End time: Wed May 13 17:12:50 2026


CalledProcessError: Command 'b'\nKEEP_FILE="/home/jupyter/repos/aou_disease_prediction_2/ckd_patients_for_plink.txt"        # e.g. "/path/to/keep.txt" with columns: FID IID (or person_id mapped to both)\nVARIANTS_FILE="/home/jupyter/repos/aou_disease_prediction_2/chr4_pkd_variants.bed"    # e.g. "/path/to/variants.txt" (1-indexed positions) OR "/path/to/regions.bed"\nPLINK_PFILE="/home/jupyter/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/pgen/exome.chr4"      # e.g. "/path/to/genotype_prefix" (expects .bed/.bim/.fam)\nPLINK_BFILE="/home/jupyter/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/bgen/chr4.bgen"      # e.g. "/path/to/genotype_prefix" (expects .bed/.bim/.fam)\n\n/opt/workbench-tools/binaries/bin/plink2 --pfile "$PLINK_PFILE" \\\n  --keep "$KEEP_FILE" \\\n  --extract bed1 "$VARIANTS_FILE" \\\n  --make-pgen \\\n  --out subset_out\n'' returned non-zero ex

In [2]:
# Build and run extraction
ensure_file(KEEP_FILE, "KEEP_FILE")
ensure_file(VARIANTS_FILE, "VARIANTS_FILE")

if not PLINK_BFILE:
    raise ValueError("Set PLINK_BFILE to your genotype prefix (.bed/.bim/.fam).")

keep2 = TMP_DIR / "keep_2col.txt"
normalize_keep_file(KEEP_FILE, keep2)

var_path = Path(VARIANTS_FILE)
ext = var_path.suffix.lower()



# if ext == ".txt":
#     ranges = TMP_DIR / "variant_ranges.txt"
#     variants_txt_to_extract_ranges(VARIANTS_FILE, CHR, ranges)
#     cmd = (
#         f"plink --bfile {shlex.quote(PLINK_BFILE)} "
#         f"--keep {shlex.quote(str(keep2))} "
#         f"--extract range {shlex.quote(str(ranges))} "
#         f"--make-bed --out {shlex.quote(OUT_PREFIX)}"
#     )
# elif ext == ".bed":
#     ids = TMP_DIR / "variant_ids.txt"
#     bed_to_variant_ids(VARIANTS_FILE, VCF_FILE, ids)
#     cmd = (
#         f"plink --bfile {shlex.quote(PLINK_BFILE)} "
#         f"--keep {shlex.quote(str(keep2))} "
#         f"--extract {shlex.quote(str(ids))} "
#         f"--make-bed --out {shlex.quote(OUT_PREFIX)}"
#     )
# else:
#     raise ValueError("VARIANTS_FILE must end with .txt or .bed")

run(cmd)
print(f"Done. Output files with prefix: {OUT_PREFIX}")


NameError: name 'PLINK_BFILE' is not defined